In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier


# ============================================================
# SETTINGS
# ============================================================

RANDOM_SEED = 42
TEST_SIZE = 0.20

DATASET_PATH = "scholarship_dataset.csv"
MODELS_DIR = "models"
METRICS_FILE = "all_model_metrics.json"


# ============================================================
# PROJECT PATH
# ============================================================

try:
    PROJECT_ROOT = Path(__file__).parent.resolve()
except NameError:
    PROJECT_ROOT = Path.cwd()

DATASET_FILE = PROJECT_ROOT / DATASET_PATH
MODELS_FOLDER = PROJECT_ROOT / MODELS_DIR
METRICS_PATH = PROJECT_ROOT / METRICS_FILE

MODELS_FOLDER.mkdir(exist_ok=True)


# ============================================================
# FEATURES
# ============================================================

FEATURE_COLUMNS = [
    "gpa",
    "family_income",
    "is_orphan",
    "is_displaced",
    "region",
    "high_school_type",
    "has_verification",
    "gender",
    "faculty"
]

CONTINUOUS_FEATURES = [
    "gpa",
    "family_income"
]

BINARY_FEATURES = [
    "is_orphan",
    "is_displaced",
    "has_verification"
]

CATEGORICAL_FEATURES = [
    "region",
    "high_school_type",
    "gender",
    "faculty"
]

TARGET_COLUMN = "Status"


# ============================================================
# PREPROCESSOR
# ============================================================

def create_preprocessor():

    return ColumnTransformer(
        transformers=[
            (
                "continuous",
                StandardScaler(),
                CONTINUOUS_FEATURES
            ),

            (
                "binary",
                "passthrough",
                BINARY_FEATURES
            ),

            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                CATEGORICAL_FEATURES
            )
        ]
    )


# ============================================================
# MODELS + PARAMETERS
# ============================================================

def create_models():

    return {

        "XGBoost": {
            "model": XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_SEED,
                n_jobs=-1
            ),

            "params": {
                "classifier__n_estimators": [100, 200, 300],
                "classifier__max_depth": [3, 5, 7],
                "classifier__learning_rate": [0.05],
                "classifier__subsample": [0.7],
                "classifier__colsample_bytree": [0.8]
            },

            "file": "xgboost_scholarship_model.pkl"
        },


        "Random Forest": {
            "model": RandomForestClassifier(
                random_state=RANDOM_SEED,
                n_jobs=-1
            ),

            "params": {
                "classifier__n_estimators": [100, 200, 300],
                "classifier__max_depth": [None, 10, 20],
                "classifier__min_samples_split": [2, 5],
                "classifier__min_samples_leaf": [1, 2]
            },

            "file": "random_forest_scholarship_model.pkl"
        },


        "Logistic Regression": {
            "model": LogisticRegression(
                random_state=RANDOM_SEED,
                max_iter=2000
            ),

            "params": {
                "classifier__C": [
                    0.01,
                    0.1,
                    1,
                    10,
                    100
                ],

                "classifier__solver": [
                    "liblinear",
                    "lbfgs"
                ]
            },

            "file": "logistic_regression_scholarship_model.pkl"
        },


        "SVM": {
            "model": SVC(
                probability=True,
                random_state=RANDOM_SEED
            ),

            "params": {
                "classifier__C": [
                    0.1,
                    1,
                    10,
                    100
                ],

                "classifier__kernel": [
                    "linear",
                    "rbf"
                ],

                "classifier__gamma": [
                    "scale",
                    "auto"
                ]
            },

            "file": "svm_scholarship_model.pkl"
        }
    }


# ============================================================
# EVALUATE MODEL
# ============================================================

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )

    confusion = confusion_matrix(
        y_test,
        y_pred
    ).tolist()

    report = classification_report(
        y_test,
        y_pred,
        target_names=[
            "Not Eligible",
            "Eligible"
        ],
        output_dict=True,
        zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "confusion_matrix": confusion,
        "classification_report": report
    }


# ============================================================
# CROSS VALIDATION
# ============================================================

def calculate_cross_validation(model, X_train, y_train, cv):

    scoring = {
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    }

    results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    return {

        "accuracy_mean":
            float(np.mean(results["test_accuracy"])),

        "accuracy_std":
            float(np.std(results["test_accuracy"])),

        "precision_mean":
            float(np.mean(results["test_precision"])),

        "precision_std":
            float(np.std(results["test_precision"])),

        "recall_mean":
            float(np.mean(results["test_recall"])),

        "recall_std":
            float(np.std(results["test_recall"])),

        "f1_mean":
            float(np.mean(results["test_f1"])),

        "f1_std":
            float(np.std(results["test_f1"]))
    }


# ============================================================
# TRAIN ONE MODEL
# ============================================================

def train_model(
    name,
    model_config,
    X_train,
    y_train,
    X_test,
    y_test,
    cv
):

    print("\n" + "=" * 70)
    print(f"{name.upper()} TRAINING")
    print("=" * 70)

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                create_preprocessor()
            ),

            (
                "classifier",
                model_config["model"]
            )
        ]
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=model_config["params"],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        verbose=1
    )

    print(f"\nTraining {name} with GridSearchCV...")

    search.fit(
        X_train,
        y_train
    )

    best_model = search.best_estimator_

    print("\nBest Parameters:")
    print(search.best_params_)

    # --------------------------------------------------------
    # HOLDOUT TEST
    # --------------------------------------------------------

    holdout = evaluate_model(
        best_model,
        X_test,
        y_test
    )

    # --------------------------------------------------------
    # CROSS VALIDATION
    # --------------------------------------------------------

    cross_validation = calculate_cross_validation(
        best_model,
        X_train,
        y_train,
        cv
    )

    # --------------------------------------------------------
    # SAVE MODEL
    # --------------------------------------------------------

    model_path = (
        MODELS_FOLDER /
        model_config["file"]
    )

    joblib.dump(
        best_model,
        model_path
    )

    # --------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------

    result = {

        "algorithm": name,

        "holdout_test": holdout,

        "cross_validation": {
            "folds": 5,
            **cross_validation
        },

        "best_parameters":
            search.best_params_,

        "model_file":
            str(model_path)
    }

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print("\n--- HOLDOUT TEST RESULTS ---")

    print(
        f"Accuracy:  "
        f"{holdout['accuracy']:.4f}"
    )

    print(
        f"Precision: "
        f"{holdout['precision']:.4f}"
    )

    print(
        f"Recall:    "
        f"{holdout['recall']:.4f}"
    )

    print(
        f"F1-score:  "
        f"{holdout['f1']:.4f}"
    )

    print(
        f"ROC-AUC:   "
        f"{holdout['roc_auc']:.4f}"
    )

    print("\n--- 5-FOLD CROSS-VALIDATION ---")

    print(
        f"Accuracy:  "
        f"{cross_validation['accuracy_mean']:.4f} "
        f"± "
        f"{cross_validation['accuracy_std']:.4f}"
    )

    print(
        f"Precision: "
        f"{cross_validation['precision_mean']:.4f} "
        f"± "
        f"{cross_validation['precision_std']:.4f}"
    )

    print(
        f"Recall:    "
        f"{cross_validation['recall_mean']:.4f} "
        f"± "
        f"{cross_validation['recall_std']:.4f}"
    )

    print(
        f"F1-score:  "
        f"{cross_validation['f1_mean']:.4f} "
        f"± "
        f"{cross_validation['f1_std']:.4f}"
    )

    print(
        f"\nModel saved to: "
        f"{model_path}"
    )

    return result


# ============================================================
# FINAL COMPARISON
# ============================================================

def print_comparison(results):

    print("\n")
    print("=" * 90)
    print("FINAL MODEL COMPARISON")
    print("=" * 90)

    rows = []

    for name, result in results.items():

        test = result["holdout_test"]

        cv = result["cross_validation"]

        rows.append({
            "Algorithm": name,

            "Test Accuracy":
                test["accuracy"],

            "Precision":
                test["precision"],

            "Recall":
                test["recall"],

            "F1":
                test["f1"],

            "ROC-AUC":
                test["roc_auc"],

            "CV Accuracy":
                cv["accuracy_mean"],

            "CV F1":
                cv["f1_mean"]
        })

    comparison = pd.DataFrame(rows)

    comparison = comparison.sort_values(
        by="CV F1",
        ascending=False
    )

    print(
        comparison.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    # --------------------------------------------------------
    # BEST MODEL
    # --------------------------------------------------------

    best_algorithm = comparison.iloc[0]["Algorithm"]

    print("\n" + "=" * 90)
    print(f"BEST MODEL: {best_algorithm}")
    print("=" * 90)

    return best_algorithm


# ============================================================
# MAIN
# ============================================================

def main():

    print("=" * 70)
    print("SCHOLARSHIP ELIGIBILITY - 4 MODEL TRAINING")
    print("=" * 70)

    print("\nLoading dataset...")

    # ========================================================
    # LOAD DATASET ONLY ONCE
    # ========================================================

    if not DATASET_FILE.exists():

        raise FileNotFoundError(
            f"Dataset not found: {DATASET_FILE}"
        )

    df = pd.read_csv(
        DATASET_FILE
    )

    print(
        f"Dataset loaded successfully: "
        f"{len(df)} records"
    )

    # ========================================================
    # VALIDATE COLUMNS
    # ========================================================

    required_columns = (
        FEATURE_COLUMNS +
        [TARGET_COLUMN]
    )

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            "Missing columns: "
            + ", ".join(missing_columns)
        )

    # ========================================================
    # DATA
    # ========================================================

    X = df[
        FEATURE_COLUMNS
    ].copy()

    y = df[
        TARGET_COLUMN
    ].astype(int)

    print(
        f"Eligible:     {int(y.sum())}"
    )

    print(
        f"Not Eligible: {int((y == 0).sum())}"
    )

    # ========================================================
    # TRAIN / TEST SPLIT ONLY ONCE
    # ========================================================

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=y
    )

    print(
        f"\nTraining samples: {len(X_train)}"
    )

    print(
        f"Testing samples:  {len(X_test)}"
    )

    # ========================================================
    # SAME CV FOR ALL MODELS
    # ========================================================

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    # ========================================================
    # TRAIN ALL FOUR MODELS
    # ========================================================

    models = create_models()

    all_results = {}

    for name, config in models.items():

        result = train_model(
            name,
            config,
            X_train,
            y_train,
            X_test,
            y_test,
            cv
        )

        all_results[name] = result

    # ========================================================
    # SAVE ALL METRICS ONCE
    # ========================================================

    output = {
        "dataset": DATASET_PATH,

        "random_seed": RANDOM_SEED,

        "test_size": TEST_SIZE,

        "number_of_samples":
            int(len(df)),

        "number_of_features":
            int(len(FEATURE_COLUMNS)),

        "models": all_results
    }

    with open(
        METRICS_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            output,
            file,
            indent=4
        )

    # ========================================================
    # FINAL COMPARISON
    # ========================================================

    best_model = print_comparison(
        all_results
    )

    print(
        f"\nAll metrics saved to: "
        f"{METRICS_PATH}"
    )

    print(
        f"\nBest model according to CV F1: "
        f"{best_model}"
    )

    print("\nTraining completed successfully!")


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    main()

SCHOLARSHIP ELIGIBILITY - 4 MODEL TRAINING

Loading dataset...
Dataset loaded successfully: 2000 records
Eligible:     622
Not Eligible: 1378

Training samples: 1600
Testing samples:  400

XGBOOST TRAINING

Training XGBoost with GridSearchCV...
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Best Parameters:
{'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 5, 'classifier__n_estimators': 300, 'classifier__subsample': 0.7}

--- HOLDOUT TEST RESULTS ---
Accuracy:  0.9750
Precision: 0.9385
Recall:    0.9839
F1-score:  0.9606
ROC-AUC:   0.9977

--- 5-FOLD CROSS-VALIDATION ---
Accuracy:  0.9925 ± 0.0042
Precision: 0.9900 ± 0.0108
Recall:    0.9859 ± 0.0103
F1-score:  0.9879 ± 0.0068

Model saved to: c:\Users\Ahmed\Desktop\scholarship-prediction-ai-main (3)\scholarship-prediction-ai-main-COMBINED-ready-to-send\scholarship-prediction-ai-main\models\xgboost_scholarship_model.pkl

RANDOM FOREST TRAINING

Training Random Forest with Gr